# Session 3: When the model learns the answers by heart

**259.075 Data-integrated Algorithmic Design Processes II** · Session 3

Last week ended with a network of sixteen kinks following a curve closely, and with a question:
what happens if you keep adding kinks until the model passes through every single point?

The answer comes in two parts, because there are two different ways for a model to be
confidently wrong. One is overfitting, which you have probably heard of. The other is
extrapolation, which is less talked about and, in this session's experiment, far worse.

You also write part of the training loop yourself today, three lines of it.

**About the code.** The variable names are German (a grading script relies on them); every name
is translated where it first appears. Cells that only draw a plot are collapsed: click the three
dots to open them if you are curious, you do not need to.

## Noise, and why it is there on purpose

Until now the target came out of a formula, and a large enough model could hit it exactly. From
here on there is random scatter on it, called **noise**. It stands in for everything a formula
cannot know about a house: who lives there, whether the windows are open in February, what the
neighbours do. Vienna publishes no measured consumption per building, so we invent the scatter:
typically 20 kWh/m²a up or down.

> Noise is not dirt in the data. It is everything the model cannot possibly predict, and the
> whole trick is not to try.

The training set is deliberately tiny, twenty-five buildings. With so few, a big model gets into
trouble quickly, and that is what we want to watch.

### How the first cell does it

1. Six `import` lines make libraries available under short names: `np` for numbers, `pd` for
   tables, `torch` and `nn` for networks, `go` and `make_subplots` for plots.
2. The cell needs random numbers, and gets them from a **random number generator** called
   `zufall` (German for chance). Think of it as a die you can ask for numbers. The `7` in
   `np.random.default_rng(7)` fixes where the die starts, so everyone gets the same numbers.
3. `df["compactness"].to_numpy()` takes the compactness column out of the table, as a plain list
   of 1073 numbers, `all_compactness`.
4. `zufall.choice(all_compactness, size=265, replace=False)` draws from that list at random, like
   names from a hat: `size=265` says how many, `replace=False` says none twice. The result is `k`,
   the compactness of our 265 buildings.
5. `rein` (German for clean) is last week's invented curve: `40 + 9 * k` is a slope, and
   `45 * np.sin(0.8 * k)` adds a wave that goes up and down by 45. No physics, just a curve.
6. `zufall.normal(0, 20, size=265)` makes 265 random numbers around 0, typically 20 up or down:
   the noise. `y`, what the model gets to see, is the clean curve plus the noise.
7. The buildings are dealt into three groups like a shuffled deck: `np.arange(count)` numbers
   them 0 to 264, `zufall.shuffle` mixes the numbers, and `i[:25]` takes the first 25 of the
   shuffled list, `i[25:105]` the next 80, `i[105:]` the rest. `len(...)` counts.

In [ ]:
# numpy: numbers and lists of numbers
import numpy as np
# pandas: tables, like a spreadsheet
import pandas as pd
# torch: the library neural networks are built with
import torch
# torch.nn: its building blocks (layers, ReLU, error measures)
import torch.nn as nn
# plotly: interactive plots
import plotly.graph_objects as go
# make_subplots: several plots side by side
from plotly.subplots import make_subplots

# The random number generator ("Zufall" = chance), starting at 7
zufall = np.random.default_rng(7)
# Our table of 1073 Vienna buildings
df = pd.read_csv("../data/buildings_vienna.csv")

# The compactness column, as a plain list of numbers
all_compactness = df["compactness"].to_numpy()
# Draw 265 of them at random: size = how many, replace=False = none twice
k = zufall.choice(all_compactness, size=265, replace=False)

# Last week's invented curve ("rein" = clean): a slope plus a wave
rein = 40 + 9 * k + 45 * np.sin(0.8 * k)
# The noise: 265 random numbers, average 0, typically 20 up or down
noise = zufall.normal(0, 20, size=265)
# What the model gets to see: clean curve plus noise
y = rein + noise

# How many buildings we have: 265
count = len(k)
# Their positions 0, 1, 2, ... 264
i = np.arange(count)
# Shuffle the positions, like a deck of cards
zufall.shuffle(i)
# Training: the first 25 of the shuffled positions
i_tr = i[:25]
# Validation: the next 80
i_va = i[25:105]
# Test: the remaining 160, locked away until the very end
i_te = i[105:]

# How many in each group
n_tr = len(i_tr)
n_va = len(i_va)
n_te = len(i_te)
print("training buildings:  ", n_tr)
print("validation buildings:", n_va)
print("test buildings:      ", n_te)

The split happens before anything is looked at. From here on, the test buildings are not drawn
and not used for any decision; they wait until the very end. (One exception, clearly marked, comes
in the homework: a single summary over all 265 buildings, to show what leakage would look like.)

### How the next cell does it

It prepares a plot of what we have. `np.concatenate` glues the training and validation positions
into one list, `sichtbar` (German for visible). To draw the clean curve from left to right, the
training buildings are sorted by compactness: `np.argsort` gives the order that would sort a list
from smallest to largest, and `i_tr[sort_order]` puts the training positions in that order,
`ordnung` (German for order). The collapsed cell after it draws the plot.

In [ ]:
# Training and validation positions together ("sichtbar" = visible)
sichtbar = np.concatenate([i_tr, i_va])
# The compactness of the 25 training buildings
k_train = k[i_tr]
# The order that sorts them from least to most compact
sort_order = np.argsort(k_train)
# The training positions in that order ("Ordnung" = order)
ordnung = i_tr[sort_order]
print("training buildings, sorted by compactness:", len(ordnung))

In [ ]:
# Plot: the clean curve (blue), validation (grey), training (red)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=k[ordnung], y=rein[ordnung], mode="lines", name="the rule",
    line=dict(width=3, color="#4c78a8")))
fig.add_trace(go.Scatter(
    x=k[i_va], y=y[i_va], mode="markers", name="validation",
    marker=dict(size=5, color="#8b95a8", opacity=0.6)))
fig.add_trace(go.Scatter(
    x=k[i_tr], y=y[i_tr], mode="markers", name="training",
    marker=dict(size=9, color="crimson")))
fig.update_layout(
    height=430, xaxis_title="compactness", yaxis_title="target",
    title="the curve underneath, and what the model gets to see")
fig.show()

Blue is the truth the model should find. The grey dots are the 80 validation buildings, the red
dots the 25 training buildings: the only ones the model learns from.

## Split first, then scale

The *training* set is what the model learns from. The *validation* set is what you look at while
deciding anything: how big the model should be, when to stop, which learning rate. The *test* set
you open once, at the very end.

The trap is in the rescaling from session 2. Average and standard deviation have to come from the
training buildings alone. Compute them over all three groups and the model has peeked at
buildings it is not supposed to have seen. That mistake is called *leakage*.

> Anything computed from the data is part of the model. That includes the average you subtracted.

### How the next cell does it

It takes the compactness and the target values of the 25 training buildings and computes from
them alone: the average (`.mean()`), the spread (`.std()`, the standard deviation from session 2),
and the smallest and largest compactness (`.min()`, `.max()`). The names follow a pattern: `k_m`
is the mean of k, `k_s` its spread, `k_lo` and `k_hi` the low and high end of the range.

At the end it counts the validation buildings beyond that range: `k_val > k_hi` gives True for
each validation building more compact than any training building, False otherwise, and `.sum()`
counts the Trues. In the print lines, `:.3f` means "show three decimals".

In [ ]:
# Compactness and target values of the 25 training buildings
k_train = k[i_tr]
y_train = y[i_tr]

# Average (_m) and spread (_s), from the training buildings only
k_m = k_train.mean()
k_s = k_train.std()
y_m = y_train.mean()
y_s = y_train.std()

# The range the model has seen: smallest and largest compactness
k_lo = k_train.min()
k_hi = k_train.max()

print(f"average compactness (training): {k_m:.3f}")
print(f"spread of compactness (training): {k_s:.3f}")
print(f"training range of compactness: {k_lo:.2f} to {k_hi:.2f}")

# Compactness of the validation buildings, and the most compact one
k_val = k[i_va]
k_val_max = k_val.max()
# True for each validation building beyond the training range
outside = k_val > k_hi
# Count the Trues
n_outside = outside.sum()
print(f"most compact validation building: {k_val_max:.2f}")
print(f"validation buildings outside the range: {n_outside}")

The 25 training buildings cover compactness from about 4.0 to 10.0, while the validation set
reaches 11.4: two validation buildings are more compact than anything the model will learn from.
Nothing unusual about that, and it matters a great deal in a moment.

## Capacity, on a dial

Before the network, a model whose flexibility sits in a single number: a **polynomial**. Degree
one is a straight line, degree three can bend twice, degree nine can bend eight times. How
flexible a model is, is called its **capacity**. Fitting a polynomial is least squares from
session 1, nothing new.

### How the next cell does it

It defines a function `polynom(grad)` (German: `Grad` = degree), which does three things:

1. `np.polyfit(x, y, grad)` finds the best curve of that degree through the training buildings
   and returns its numbers, `c`. The compactness goes in rescaled, as in session 2, because high
   powers of large numbers get out of hand.
2. A small function inside it, `vor` (short for `Vorhersage`, prediction), gives the curve's value
   for any list of compactness values (`werte` = values), using `np.polyval(c, x)`.
3. It measures the error (MSE, square the misses and average them, `**` means "to the power of")
   four times: on the training buildings, on all validation buildings, on the validation
   buildings inside the training range, and on those outside it. `drinnen` (inside) is True for
   the buildings inside: `&` means "and", `~drinnen` means "not inside".

It hands everything back in a **dictionary**, a small labelled box: `e["train"]` is the training
error, `e["val_innen"]` the error inside (`innen`), `e["val_aussen"]` outside (`aussen`),
`e["vor"]` the prediction function. `float(...)` makes each error a plain decimal number.

In [ ]:
def polynom(grad):
    """The best curve of degree `grad` through the training buildings,
    and its errors. ("grad" = degree)"""
    # Compactness of the training buildings, rescaled
    x_train = (k[i_tr] - k_m) / k_s
    # Least squares: the numbers of the best curve of this degree
    c = np.polyfit(x_train, y[i_tr], grad)

    def vor(werte):
        # The curve's value at the given compactness values
        # ("vor" = Vorhersage = prediction, "werte" = values)
        x = (werte - k_m) / k_s
        return np.polyval(c, x)

    # Validation buildings: compactness and targets
    k_val = k[i_va]
    y_val = y[i_va]
    # True for those inside the training range ("drinnen" = inside)
    drinnen = (k_val >= k_lo) & (k_val <= k_hi)

    # Predictions for each group
    p_train = vor(k[i_tr])
    p_val = vor(k_val)
    p_in = vor(k_val[drinnen])
    p_out = vor(k_val[~drinnen])
    # The four errors (MSE): square the misses, average them
    train = np.mean((p_train - y[i_tr]) ** 2)
    val = np.mean((p_val - y_val) ** 2)
    val_innen = np.mean((p_in - y_val[drinnen]) ** 2)
    val_aussen = np.mean((p_out - y_val[~drinnen]) ** 2)

    # A labelled box with the prediction function and the errors
    result = {}
    result["vor"] = vor
    result["train"] = float(train)
    result["val"] = float(val)
    result["val_innen"] = float(val_innen)
    result["val_aussen"] = float(val_aussen)
    return result


# Try it once: a straight line (degree 1)
e = polynom(1)
print(f"degree 1: training error {e['train']:.0f}")

The collapsed cell below draws four degrees, 1, 3, 6 and 9, in a 2 by 2 grid: training buildings
red, validation grey, the fitted curve orange, the training range between dotted lines. Above each
plot: its training error and its validation error inside the range.

In [ ]:
# Four degrees in a 2 by 2 grid
grade = [1, 3, 6, 9]
# 400 evenly spaced compactness values to draw smooth curves
# ("gitter" = grid, "grenze" = boundary)
k_low_end = k[sichtbar].min()
k_high_end = k[sichtbar].max()
gitter_k = np.linspace(k_low_end, k_high_end, 400)
y_low_end = y[sichtbar].min()
y_high_end = y[sichtbar].max()

titles = []
for grad in grade:
    titles.append(f"degree {grad}")
fig = make_subplots(rows=2, cols=2, subplot_titles=titles)
for n in range(4):
    grad = grade[n]
    e = polynom(grad)
    # Row and column: plots 0, 1 on top, 2, 3 below
    r = n // 2 + 1
    c = n % 2 + 1
    fig.add_trace(go.Scatter(
        x=k[i_va], y=y[i_va], mode="markers", showlegend=False,
        marker=dict(size=4, color="#8b95a8", opacity=0.5)),
        row=r, col=c)
    fig.add_trace(go.Scatter(
        x=k[i_tr], y=y[i_tr], mode="markers", showlegend=False,
        marker=dict(size=7, color="crimson")), row=r, col=c)
    curve = e["vor"](gitter_k)
    fig.add_trace(go.Scatter(
        x=gitter_k, y=curve, mode="lines", showlegend=False,
        line=dict(width=3, color="#d8a24a")), row=r, col=c)
    for grenze in (k_lo, k_hi):
        fig.add_vline(x=grenze, line_dash="dot", line_color="#556",
                      row=r, col=c)
    fig.layout.annotations[n].text = (
        f"degree {grad}: train {e['train']:.0f}, "
        f"validation inside {e['val_innen']:,.0f}")
fig.update_layout(
    height=660,
    title="25 training buildings in red, 80 validation buildings in "
          "grey, training range between the dotted lines")
fig.update_yaxes(range=[y_low_end - 20, y_high_end + 20])
fig.update_xaxes(range=[k_low_end - 0.2, k_high_end + 0.2])
fig.show()

### How the next cell does it

The same comparison as a table, for six degrees. A `for` loop runs `polynom` once per degree and
prints one line each: the degree, the training error, the validation error inside the training
range, and outside it. In the print line, `:>10` only sets how wide a column is, `:.0f` rounds to
whole numbers, and `,` adds thousands separators.

In [ ]:
# The header of the table
print(f"{'degree':>6} {'train':>10} {'val inside':>12}"
      f" {'val outside':>18}")
# One line per degree
for grad in [1, 3, 5, 6, 9, 12]:
    # Fit the curve, get its errors ("e" = Ergebnis = result)
    e = polynom(grad)
    print(f"{grad:>6} {e['train']:>10.0f} {e['val_innen']:>12.0f}"
          f" {e['val_aussen']:>18,.0f}")
print()
print("inside = the 78 validation buildings within the training range")
print("outside = the 2 beyond it")

Two separate things are going wrong, and the table separates them.

Inside the training range, the validation error is lowest at degree six (497, with degree five
close behind at 502) and then climbs: to 1059 at degree nine and over 20 000 at degree twelve.
That is **overfitting**. The curve spends its flexibility on the noise of twenty-five particular
buildings, and on other buildings drawn the same way it does worse.

Outside the training range, at the two buildings with compactness above ten, degree nine has a
mean squared error of over two billion: a typical miss of about 48 000 on values around a hundred.
That is **extrapolation**, predicting beyond what the model has seen. A polynomial has no opinion
about territory it has not seen; out there it does whatever its highest power happens to do.

> In this experiment, overfitting cost a factor of two to forty. Extrapolation cost everything,
> and nothing in the output warns you which one you are doing.

The training error, meanwhile, falls with every degree and never turns around.

> The training error is not a measure of quality. It is a measure of how hard the model tried.

## The same thing, with a network

A network has no single dial. Its capacity sits in the width (neurons per layer), the number of
layers and the activation function, all fixed before training starts. How long you train does not
change what it could represent, but it changes which curve you end up with. We build it in four
small cells.

### Cell 1: numbers for PyTorch

PyTorch computes with its own kind of list, the **tensor**. The function `als_tensor` (German: "as
a tensor") rescales values as in session 2 (`mitte` = average, `spanne` = spread), turns them
into a tensor of decimal numbers (`dtype=torch.float32`), and stands them upright as one column
(`unsqueeze(1)`), because a network wants one row per building. Then it makes the six tensors we
need: inputs `X` (compactness) and targets `Y`, for training (`tr`), validation (`va`) and test
(`te`).

In [ ]:
def als_tensor(werte, mitte, spanne):
    """Numbers ready for PyTorch: rescaled, a tensor, one column.
    ("als Tensor" = as a tensor, "werte" = values,
    "mitte" = average, "spanne" = spread)"""
    # Rescale: subtract the average, divide by the spread
    scaled = (werte - mitte) / spanne
    # PyTorch's own kind of list, the tensor, of decimal numbers
    tensor = torch.tensor(scaled, dtype=torch.float32)
    # Stand it upright: one column, one row per building
    return tensor.unsqueeze(1)


# Inputs X (compactness) and targets Y, for each of the three groups
Xtr = als_tensor(k[i_tr], k_m, k_s)
Ytr = als_tensor(y[i_tr], y_m, y_s)
Xva = als_tensor(k[i_va], k_m, k_s)
Yva = als_tensor(y[i_va], y_m, y_s)
Xte = als_tensor(k[i_te], k_m, k_s)
Yte = als_tensor(y[i_te], y_m, y_s)
print("training inputs, as rows and columns:", tuple(Xtr.shape))

`(25, 1)`: 25 rows, one per training building, and one column.

### Cell 2: the network

`baue` (German: build) makes a network. `nn.Linear(1, neuronen)` is a layer of `neuronen`
(neurons) that each take the one input with their own weight and bias; `nn.ReLU()` puts the kink
behind each, as in session 2. A second layer of the same kind follows, and `nn.Linear(neuronen,
1)` adds everything up into one output. `nn.Sequential` strings the layers together in that order.
`samen` (seed) fixes the random starting weights, so everyone gets the same network.

In [ ]:
def baue(neuronen=64, samen=0):
    """A network with `neuronen` neurons per layer.
    ("baue" = build, "neuronen" = neurons, "samen" = seed)"""
    # Fix the random starting weights
    torch.manual_seed(samen)
    # Layer 1: one input to `neuronen` ramps
    layer_1 = nn.Linear(1, neuronen)
    # Layer 2: those ramps to `neuronen` new ramps
    layer_2 = nn.Linear(neuronen, neuronen)
    # Output: everything added up into one number
    layer_out = nn.Linear(neuronen, 1)
    # String them together, with a ReLU kink after layers 1 and 2
    return nn.Sequential(layer_1, nn.ReLU(), layer_2, nn.ReLU(),
                         layer_out)


# Build one, to see it
probe_net = baue(4)
print(probe_net)

### Cell 3: the learning loop, and your part

`trainiere` (German: train) takes a network (`modell` = model) and trains it for `schritte`
(steps) at a `lernrate` (learning rate). Before the loop it sets up two things:

- `verlust` (loss): the error measure, MSE as in session 1.
- `optimierer` (optimiser): the thing that does the step downhill. We use **Adam**, a smarter
  version of "weight minus learning rate times gradient" that adapts the step for every weight.

Then, for every step, five lines:

    vorhersage = modell(Xtr)            the network predicts the training buildings
    fehler = verlust(vorhersage, Ytr)   how far off, as one number
    optimierer.zero_grad()              throw away the previous step's gradients
    fehler.backward()                   compute this step's gradients
    optimierer.step()                   move every weight a little downhill

(`vorhersage` = prediction, `fehler` = error.) Every 25 steps (`alle` = every) it also measures:
`getan % alle == 0` is True when the number of steps done (`getan`) divides by 25 without
remainder. Inside `with torch.no_grad():` PyTorch only computes and does not prepare any learning,
which is right for measuring. It writes down when it measured (`stellen`, positions), the training
and validation error (`kurve_tr`, `kurve_va`, curves), and a copy of all weights at that moment
(`zustaende`, states): `state_dict()` lists every weight by name, and `.detach().clone()` makes an
independent copy that later steps do not change.

At the end it hands everything back in one labelled box, a dictionary.

**Your part:** in the loop, the last two of the five lines are already there. Replace the line
`raise NotImplementedError(...)` with the first three lines of the list above, exactly as written,
at the same indentation. If `zero_grad()` is missing, PyTorch adds each new gradient to the old
ones instead of replacing them, and the model walks off in a direction that is the sum of
everything it ever wanted to do. Stuck for ten minutes? The solution is released, see
`kursregeln.md`.

In [ ]:
def trainiere(modell, schritte=6000, lernrate=0.01, alle=25):
    """The learning loop. Measures every `alle` steps and keeps a copy
    of the weights each time. ("trainiere" = train, "modell" = model,
    "schritte" = steps, "lernrate" = learning rate, "alle" = every)"""
    # The error measure: MSE ("Verlust" = loss)
    verlust = nn.MSELoss()
    # All weights of the network
    weights = modell.parameters()
    # The optimiser does the steps downhill (Adam)
    optimierer = torch.optim.Adam(weights, lr=lernrate)
    # Empty lists for the records ("stellen" = positions,
    # "kurve" = curve, "zustaende" = states)
    stellen = []
    kurve_tr = []
    kurve_va = []
    zustaende = []

    # "schritt" = step
    for schritt in range(schritte):
        # YOUR TURN: replace the raise line below with three lines:
        #   vorhersage = modell(Xtr)
        #   fehler = verlust(vorhersage, Ytr)
        #   optimierer.zero_grad()
        raise NotImplementedError("training loop missing")
        # Compute the gradient of every weight
        fehler.backward()
        # Move every weight a little downhill
        optimierer.step()

        # Number of steps done so far ("getan" = done)
        getan = schritt + 1
        # Every 25 steps, and at the very end: measure
        if getan % alle == 0 or getan == schritte:
            # Measuring only, nothing is learned here
            with torch.no_grad():
                stellen.append(getan)
                # Error on the training buildings
                p_tr = modell(Xtr)
                e_tr = verlust(p_tr, Ytr)
                kurve_tr.append(float(e_tr))
                # Error on the validation buildings
                p_va = modell(Xva)
                e_va = verlust(p_va, Yva)
                kurve_va.append(float(e_va))
                # An independent copy of every weight ("Kopie")
                kopie = {}
                all_weights = modell.state_dict()
                for name in all_weights:
                    werte = all_weights[name]
                    kopie[name] = werte.detach().clone()
                zustaende.append(kopie)

    # Hand everything back in one labelled box
    records = {}
    records["stellen"] = np.array(stellen)
    records["tr"] = np.array(kurve_tr)
    records["va"] = np.array(kurve_va)
    records["zustaende"] = zustaende
    return records

<details>
<summary>Answer: the three lines</summary>

    vorhersage = modell(Xtr)
    fehler = verlust(vorhersage, Ytr)
    optimierer.zero_grad()

</details>

### Cell 4: four widths

Now four networks, with 2, 4, 16 and 64 neurons per layer (`breite` = width), each trained for
6000 steps. That takes about half a minute. For each, `np.argmin` finds the position of the
lowest validation error, stored as `tief` (German: low). All runs go into `laeufe` (runs), a
dictionary with the width as label. Then it prints how the widest network did. `[-1]` means "the
last entry".

In [ ]:
# All runs, labelled by width ("Läufe" = runs)
laeufe = {}
# "Breite" = width: neurons per layer
for breite in (2, 4, 16, 64):
    # Build a network of this width
    m = baue(breite)
    # Train it for 6000 steps
    records = trainiere(m)
    # Position of the lowest validation error ("tief" = low)
    lowest = np.argmin(records["va"])
    records["tief"] = int(lowest)
    # Keep the run
    laeufe[breite] = records

# Look at the widest network ("Lauf" = run)
lauf = laeufe[64]
tief = lauf["tief"]
best_val = lauf["va"][tief]
best_step = lauf["stellen"][tief]
end_val = lauf["va"][-1]
end_train = lauf["tr"][-1]
last_step = lauf["stellen"][-1]
print("64 neurons per layer:")
print(f"  lowest validation error {best_val:.3f} after {best_step} updates")
print(f"  after all {last_step} updates: {end_val:.3f}")
print(f"  training error at the end: {end_train:.3f}")

The collapsed cell below draws the two error curves of the 64-neuron network over the training
steps: training in blue, validation in red, a dashed line at the lowest validation error. Both
axes are logarithmic, each tick ten times the one before, so the early steps get room.

In [ ]:
# Error curves of the 64-neuron network, log axes
stellen = lauf["stellen"]
kurve_tr = lauf["tr"]
kurve_va = lauf["va"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=stellen, y=kurve_tr, mode="lines",
                         name="training",
                         line=dict(width=3, color="#4c78a8")))
fig.add_trace(go.Scatter(x=stellen, y=kurve_va, mode="lines",
                         name="validation",
                         line=dict(width=3, color="crimson")))
fig.add_vline(x=stellen[tief], line_dash="dash", line_color="#d8a24a")
# Text labels on log axes are placed by the logarithm of the position
label_x = np.log10(stellen[tief])
label_y = np.log10(kurve_va.max())
fig.add_annotation(x=label_x, y=label_y, text="stop here",
                   showarrow=False, xshift=45,
                   font=dict(color="#d8a24a"))
fig.update_layout(height=470, xaxis_title="updates",
                  yaxis_title="error, in scaled units",
                  yaxis_type="log", xaxis_type="log",
                  title="64 neurons per layer, 25 buildings to learn from")
fig.show()

The training error falls overall, though not on every interval: Adam is not least squares, and a
step that helps on average can make things briefly worse. The validation error falls, bottoms
out, and rises. The dashed line marks its lowest point.

Say it carefully: that point estimates when you should have stopped, on this validation set. It
does not mark a moment at which learning turned into memorising; both were mixed from the first
step. And notice how much gentler the rise is than the polynomial's: here the validation error
rises from its best (0.293) to the end (0.377), by about a third; the polynomial's rose from
degree six to degree twelve by a factor of forty.

The collapsed cell below walks through the same run with a slider. Left the two curves up to that
moment, right what the network's curve looks like on the data at that moment. Drag it.

In [ ]:
# Slider through the training run
# The compactness grid, ready for PyTorch ("gitter" = grid)
gitter = als_tensor(gitter_k, k_m, k_s)
# About 25 moments to show ("proben" = samples), plus the last one
step = max(1, len(stellen) // 25)
proben = list(range(0, len(stellen), step))
if len(stellen) - 1 not in proben:
    proben.append(len(stellen) - 1)

# The network's curve at each moment ("hilfs" = helper, "linien" = lines)
hilfs = baue()
linien = []
for j in proben:
    hilfs.load_state_dict(lauf["zustaende"][j])
    with torch.no_grad():
        p = hilfs(gitter).numpy().ravel()
    linien.append(p * y_s + y_m)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("error so far", "the model on the data"))
fig.add_trace(go.Scatter(x=stellen[:1], y=kurve_tr[:1],
                         mode="lines+markers",
                         line=dict(width=3, color="#4c78a8")), row=1, col=1)
fig.add_trace(go.Scatter(x=stellen[:1], y=kurve_va[:1],
                         mode="lines+markers",
                         line=dict(width=3, color="crimson")), row=1, col=1)
fig.add_trace(go.Scatter(x=k[i_va], y=y[i_va], mode="markers",
                         marker=dict(size=4, color="#8b95a8", opacity=0.5)),
              row=1, col=2)
fig.add_trace(go.Scatter(x=k[i_tr], y=y[i_tr], mode="markers",
                         marker=dict(size=8, color="crimson")), row=1, col=2)
fig.add_trace(go.Scatter(x=gitter_k, y=linien[0], mode="lines",
                         line=dict(width=3, color="#d8a24a")), row=1, col=2)
frames = []
slider_steps = []
for n in range(len(proben)):
    j = proben[n]
    frames.append(go.Frame(
        name=str(stellen[j]), traces=[0, 1, 4],
        data=[go.Scatter(x=stellen[:j + 1], y=kurve_tr[:j + 1],
                         mode="lines+markers",
                         line=dict(width=3, color="#4c78a8")),
              go.Scatter(x=stellen[:j + 1], y=kurve_va[:j + 1],
                         mode="lines+markers",
                         line=dict(width=3, color="crimson")),
              go.Scatter(x=gitter_k, y=linien[n], mode="lines",
                         line=dict(width=3, color="#d8a24a"))]))
    slider_steps.append(dict(
        method="animate", label=str(stellen[j]),
        args=[[str(stellen[j])],
              dict(mode="immediate", frame=dict(duration=0, redraw=False))]))
fig.frames = frames
fig.update_layout(height=520, showlegend=False,
                  sliders=[dict(active=0, steps=slider_steps,
                                currentvalue=dict(prefix="updates: "))])
lowest = min(kurve_tr.min(), kurve_va.min())
highest = max(kurve_tr.max(), kurve_va.max())
fig.update_xaxes(title_text="updates", type="log",
                 range=[0, np.log10(stellen[-1])], row=1, col=1)
fig.update_yaxes(title_text="error", type="log",
                 range=[np.log10(lowest * 0.8), np.log10(highest * 1.2)],
                 row=1, col=1)
fig.update_xaxes(title_text="compactness",
                 range=[k_low_end - 0.2, k_high_end + 0.2], row=1, col=2)
fig.update_yaxes(title_text="target",
                 range=[y_low_end - 20, y_high_end + 20], row=1, col=2)
fig.show()

## Two ways out, and how much each is worth

The first is to use less model. On the polynomial that worked well: degree five beat degree nine
by a factor of two inside the training range (502 against 1059), and outside it by about six orders
of magnitude (1146 against 2.3 billion).

The second is to stop at the lowest point of the validation curve instead of at the end. That is
what the copies of the weights in `trainiere` are for, and taking the copy from the best moment is
called **early stopping**.

### How the next cell does it

The collapsed part draws, for each of the four widths, the curve of its network at its best
moment. The visible part then prints a table, one line per width: the best validation error, at
which update, the error at the end, and how much stopping early saved, as a percentage of the
error at the end: (end − best) / end × 100 (`gewinn` = gain).

In [ ]:
# Each width at its best moment, in a 2 by 2 grid ("Breiten" = widths)
breiten = [2, 4, 16, 64]
titles = []
for breite in breiten:
    titles.append(f"{breite} neurons per layer")
fig = make_subplots(rows=2, cols=2, subplot_titles=titles)
for n in range(4):
    breite = breiten[n]
    e = laeufe[breite]
    hilfs = baue(breite)
    hilfs.load_state_dict(e["zustaende"][e["tief"]])
    with torch.no_grad():
        p = hilfs(gitter).numpy().ravel()
    linie = p * y_s + y_m
    r = n // 2 + 1
    c = n % 2 + 1
    fig.add_trace(go.Scatter(x=k[i_tr], y=y[i_tr], mode="markers",
                             showlegend=False,
                             marker=dict(size=7, color="crimson")),
                  row=r, col=c)
    fig.add_trace(go.Scatter(x=gitter_k, y=linie, mode="lines",
                             showlegend=False,
                             line=dict(width=3, color="#d8a24a")),
                  row=r, col=c)
fig.update_layout(height=640, title="four widths, each stopped where "
                                    "its validation error was lowest")
fig.update_yaxes(range=[y_low_end - 20, y_high_end + 20])
fig.update_xaxes(range=[k_low_end - 0.2, k_high_end + 0.2])
fig.show()

In [ ]:
# The header of the table
print(f"{'width':>6} {'best':>8} {'at update':>11} {'at the end':>12}"
      f" {'saved by stopping':>19}")
for breite in breiten:
    e = laeufe[breite]
    # Position of the best moment
    j = e["tief"]
    best = e["va"][j]
    at_update = e["stellen"][j]
    end = e["va"][-1]
    # How much stopping early saved, in percent ("Gewinn" = gain)
    gewinn = 100 * (end - best) / end
    print(f"{breite:>6} {best:>8.3f} {at_update:>11}"
          f" {end:>12.3f} {gewinn:>18.1f}%")

Read the numbers before you conclude anything. In this run, four, sixteen and sixty-four neurons
land within about five percent of each other, close enough that a different random split could
reorder them. Only two neurons is clearly behind, and its problem looks like the opposite one: its
curve stays too stiff to follow both turns. You cannot tell by looking at the architecture; the
validation set decides, not the recipe.

Early stopping is worth between 2.5 and 22.3 percent here, depending on the width. Be honest about
that number: we picked the best of 240 saved moments using the validation set, so an improvement
over the last one was close to guaranteed. Whether it also helps on genuinely new data, the
validation set cannot answer. That is what the test set is for.

## Opening the test set

Once, now, together. The rule is stated before the number is read: take the width with the lowest
validation error, at its own best moment, and apply it to the 160 buildings nobody has touched.

<img src="img/wachssiegel.jpg" width="230">

*Photo: postal authorities of the Russian Empire, 1855,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Reval_1855-02-09_church_letter_wax_seal_(cropped).jpg),
public domain.*

### How the next cell does it

1. A loop goes through the four widths and keeps the one whose best validation error is the
   smallest so far: `gewaehlt` (German: chosen).
2. A fresh network of that width, `bestes` (the best), gets the weights from its best moment.
3. It computes the error on the validation and, once, on the test buildings. `nn.MSELoss()`
   makes the error measure, which is then used like a function.
4. The errors are in rescaled units. `np.sqrt(error) * y_s` turns them into a typical miss in
   kWh/m²a: square root back to plain units, times the spread to undo the rescaling.
5. The last lines compute the typical miss of the clean curve `rein` itself, against the noisy
   values: the noise we put in, which no model can predict.

In [ ]:
# Find the width with the lowest best validation error
gewaehlt = None
smallest = None
for breite in breiten:
    e = laeufe[breite]
    best = e["va"][e["tief"]]
    # The first width, or a better one than so far
    if smallest is None or best < smallest:
        smallest = best
        gewaehlt = breite

# A fresh network of that width, with its best weights
e = laeufe[gewaehlt]
bestes = baue(gewaehlt)
best_weights = e["zustaende"][e["tief"]]
bestes.load_state_dict(best_weights)
print(f"chosen by validation: {gewaehlt} neurons,"
      f" stopped after {e['stellen'][e['tief']]} updates")

# The error measure, used like a function
mse = nn.MSELoss()
with torch.no_grad():
    # Error on the validation buildings
    p_va = bestes(Xva)
    fehler_va = float(mse(p_va, Yva))
    # Error on the test buildings: opened now, once
    p_te = bestes(Xte)
    fehler_te = float(mse(p_te, Yte))

# Typical miss in kWh/m2a: square root, times the spread of y
miss_va = np.sqrt(fehler_va) * y_s
miss_te = np.sqrt(fehler_te) * y_s
print(f"validation: {fehler_va:.3f} scaled, typical miss {miss_va:.1f} kWh/m2a")
print(f"test      : {fehler_te:.3f} scaled, typical miss {miss_te:.1f} kWh/m2a")

# The clean curve itself against the noisy values: just the noise
miss_rule_va = np.sqrt(np.mean((rein[i_va] - y[i_va]) ** 2))
miss_rule_te = np.sqrt(np.mean((rein[i_te] - y[i_te]) ** 2))
print(f"the clean curve on validation: {miss_rule_va:.1f} kWh/m2a")
print(f"the clean curve on test:       {miss_rule_te:.1f} kWh/m2a")

# The compactness range of each group
for name, idx in (("training", i_tr), ("validation", i_va),
                  ("test", i_te)):
    low = k[idx].min()
    high = k[idx].max()
    print(f"compactness in the {name} buildings: {low:.2f} to {high:.2f}")

The typical miss lands near the twenty we put in, and the clean curve explains why: even the exact
rule that generated the data misses by about nineteen on these buildings, because the noise is
still there. Twenty is what to expect on average over many fresh buildings; a lucky group of
eighty can score below it.

What the test number means: an estimate of how this model does on new buildings drawn the same
way. Not: anything about another district or construction period. A few test buildings lie outside
the training range, so the number contains some extrapolation, but it does not show how the model
does on those alone.

> You get one look at the test set. Tune anything after looking at it and you no longer have one.

## Tests

This cell has to run green before you submit. Each `assert` checks one thing and stops with the
message after it if that thing is not true. Four checks:

1. **The weights move.** A small network is copied before and after 600 steps; `torch.equal`
   tells whether two tensors are identical, and at least one weight must have changed. The errors
   must stay finite numbers (`np.isfinite`), and the training error must fall.
2. **The gradients are cleared.** After two steps, the gradient stored in the network must equal
   the gradient computed fresh for the second step alone. If `zero_grad()` is missing, the two
   steps' gradients are added up and differ.
3. **Validation does not teach.** `trainiere` uses `Yva` only for measuring. The test changes the
   validation targets by 500, trains an identical network again, and the weights must come out
   exactly the same. Then it puts the real targets back (`echt` = real).
4. **Better than a straight line.** The network's best validation error must beat a straight line
   (`gerade`), whose error is rescaled the same way, divided by `y_s` squared.

In [ ]:
# 1. The weights move ("Probe" = test, "vorher" = before)
probe = baue(16, samen=5)
vorher = []
for weight in probe.parameters():
    vorher.append(weight.detach().clone())
rec_probe = trainiere(probe, schritte=600)
nachher = list(probe.parameters())
moved = False
for n in range(len(vorher)):
    if not torch.equal(vorher[n], nachher[n]):
        moved = True
assert moved, "the weights did not move; is optimierer.step() there?"
finite_tr = np.isfinite(rec_probe["tr"]).all()
finite_va = np.isfinite(rec_probe["va"]).all()
assert finite_tr and finite_va, "the error became inf or nan"
assert rec_probe["tr"][-1] < rec_probe["tr"][0], "training error must fall"

# 2. The gradients are cleared ("zwei" = two, "Kontrolle" = control)
zwei = baue(16, samen=5)
rec_zwei = trainiere(zwei, schritte=2, alle=1)
kontrolle = baue(16, samen=5)
# The weights after the first step
kontrolle.load_state_dict(rec_zwei["zustaende"][0])
# The gradient of the second step, computed fresh
p_k = kontrolle(Xtr)
loss_k = nn.MSELoss()(p_k, Ytr)
loss_k.backward()
grads_zwei = list(zwei.parameters())
grads_kontrolle = list(kontrolle.parameters())
for n in range(len(grads_zwei)):
    g1 = grads_zwei[n].grad
    g2 = grads_kontrolle[n].grad
    assert g1 is not None, "no gradients; is fehler.backward() there?"
    same = torch.allclose(g1, g2, atol=1e-5)
    assert same, "two steps' gradients were added; zero_grad() missing?"

# 3. Validation does not teach ("echt" = real)
Yva_echt = Yva.clone()
net_a = baue(16, samen=9)
trainiere(net_a, schritte=120)
# Change the validation targets wildly, train an identical network
Yva += 500.0
net_b = baue(16, samen=9)
trainiere(net_b, schritte=120)
# Put the real validation targets back
Yva.copy_(Yva_echt)
weights_a = list(net_a.parameters())
weights_b = list(net_b.parameters())
for n in range(len(weights_a)):
    same = torch.allclose(weights_a[n], weights_b[n])
    assert same, "changing validation targets changed the weights"

# 4. Better than a straight line ("Gerade" = straight line)
gerade = np.polyfit(k[i_tr], y[i_tr], 1)
p_line = np.polyval(gerade, k[i_va])
line_error = np.mean((p_line - y[i_va]) ** 2)
# Rescaled like the network's error
rest_gerade = float(line_error / y_s ** 2)
best_probe = rec_probe["va"].min()
assert best_probe < rest_gerade, (
    f"a straight line does better than your network:"
    f" {best_probe:.3f} against {rest_gerade:.3f}")

print(f"all green: best validation {best_probe:.3f},"
      f" straight line {rest_gerade:.3f}")

## Where you stand now

You can write a training loop in PyTorch and say what each line does. You can split a dataset into
three parts in the right order and explain why the rescaling comes out of one of them. You can read
a pair of error curves and name the point at which you should have stopped, knowing that the point
was chosen using the same data it is judged on. And you can tell overfitting from extrapolation,
two different failures that, in this experiment, had very different sizes.

Next session the input stops being a number somebody chose for you. It becomes the floor plan.

---

### Terms to learn

| term | in one sentence |
|---|---|
| overfitting | the model fits its training data ever better and new data worse; often seen as validation error rising while training error falls |
| extrapolation | predicting outside the range the model was fitted on; here far worse than overfitting |
| capacity | how much a model can represent; degree, width and depth feed it |
| training set | what the model learns from |
| validation set | what you look at while deciding anything about the model |
| test set | opened once, at the end; an estimate for new data of the same kind |
| leakage | information from held-out data reaching the model, for example through scaling |
| early stopping | keeping the weights from the best validation moment instead of the last |
| regularisation | anything that trades training accuracy for behaviour on new data |
| expected error on fresh data | what the noise level predicts on average; a single sample can beat it by luck |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

Set the network up on the noisy data so that the validation error is as small as you can get it
within a fixed experiment, freeze it at its best point, and submit it.

1. Choose two learning rates between 0.001 and 0.05 and a seed of your own. The first cell trains
   three widths (4, 16 and 64 neurons per layer) at each of your rates, six runs of 6000 updates,
   about twenty seconds. You do not have to search further.
2. The second cell takes the run with the lowest validation error, shows its curves, and saves it
   as `abgabe_03.pt`. Submit that file with the notebook.

The training and validation buildings stay as they are. The test buildings stay shut: when you
rerun the whole notebook, the class cell that opened them prints its number again; that is the one
look from class, and nothing you decide may depend on it.

Your submitted model is graded on a further set of houses that nobody has seen, drawn the same way.
A particularly good number there earns no point; what earns a point is that you predicted it
sensibly. A flat validation curve is a legitimate observation and costs nothing, as long as you
show it and read it correctly.

### How the first homework cell does it

It checks your choices with a few `assert` lines (`None not in lernraten` checks that both rates
are filled in, `isinstance(mein_samen, int)` that the seed is a whole number). Then two loops
inside each other: for each of your two learning rates, for each of the three widths, it builds a
network with your seed and trains it. If a run blows up, the error becomes infinite (`inf`) or not
a number (`nan`), which happens when the learning rate is too large; `continue` then skips to the
next run. Otherwise it writes the best moment of the run into a log, `protokoll`.

At the end it prints the rescaling from the training buildings next to the rescaling over all 265
buildings, test included. That one summary deliberately breaks the rule, to show you for question 3
what leakage would have looked like; nothing is decided with it.

In [ ]:
# The three widths to try ("Breiten" = widths)
breiten = [4, 16, 64]

# YOUR CHOICE: two learning rates between 0.001 and 0.05 ("Lernraten"),
# and a whole number as your seed ("mein Samen" = my seed)
lernraten = [None, None]
mein_samen = None

# Checks on your choices
assert len(lernraten) == 2, "enter exactly two learning rates"
assert None not in lernraten, "enter both learning rates"
assert mein_samen is not None, "enter a seed"
for lr in lernraten:
    assert 0.001 <= lr <= 0.05, "both rates between 0.001 and 0.05"
assert lernraten[0] != lernraten[1], "two different rates"
assert isinstance(mein_samen, int), "the seed is a whole number"

# One entry per run ("Protokoll" = log)
protokoll = []
for lr in lernraten:
    for n in breiten:
        # A network of width n, with your seed
        m = baue(n, samen=mein_samen)
        # Train it, 6000 steps, at this learning rate
        records = trainiere(m, schritte=6000, lernrate=lr)
        kva = records["va"]
        # The error blew up: skip this run
        if not np.isfinite(kva).all():
            print(f"{n:4d} neurons, learning rate {lr:<6}"
                  f" ran away (inf or nan), left out")
            continue
        # The best moment of this run
        j = int(np.argmin(kva))
        st_j = int(records["stellen"][j])
        # Write it into the log ("Eintrag" = entry, "Stelle" = position,
        # "Zustand" = state, the weights at that moment)
        eintrag = {}
        eintrag["neuronen"] = n
        eintrag["lernrate"] = lr
        eintrag["stelle"] = st_j
        eintrag["val"] = float(kva[j])
        eintrag["zustand"] = records["zustaende"][j]
        eintrag["stellen"] = records["stellen"]
        eintrag["tr"] = records["tr"]
        eintrag["va"] = kva
        protokoll.append(eintrag)
        print(f"{n:4d} neurons, learning rate {lr:<6}"
              f" best validation {kva[j]:.4f} after {st_j:5d} updates,"
              f" at the end {kva[-1]:.3f}")
    # An empty line between the two learning rates
    print()

# The rescaling from training only, and (deliberately) from all 265
all_mean = k.mean()
all_spread = k.std()
print(f"scaling from training only: mean {k_m:.3f}, spread {k_s:.3f}")
print(f"scaling from all 265      : mean {all_mean:.3f},"
      f" spread {all_spread:.3f}")

### How the second homework cell does it

A loop goes through the log and keeps the run with the lowest validation error, `wahl` (German:
choice). The collapsed part draws its two error curves, with a dashed line where it was frozen.
Then it prints its error as a typical miss in kWh/m²a, next to what the clean curve itself misses,
and `torch.save` writes the chosen network, with the averages and spreads needed to use it, into
the file `abgabe_03.pt` (`Abgabe` = submission).

In [ ]:
# At least one run must have worked
assert protokoll, "every run ran away; choose smaller learning rates"
# Keep the run with the lowest validation error ("Wahl" = choice)
wahl = None
for eintrag in protokoll:
    if wahl is None or eintrag["val"] < wahl["val"]:
        wahl = eintrag
print(f"your choice: {wahl['neuronen']} neurons, learning rate"
      f" {wahl['lernrate']}, frozen after {wahl['stelle']} updates")

In [ ]:
# The chosen run's two error curves, log axes
fig = go.Figure()
fig.add_trace(go.Scatter(x=wahl["stellen"], y=wahl["tr"],
                         mode="lines", name="training",
                         line=dict(width=3, color="#4c78a8")))
fig.add_trace(go.Scatter(x=wahl["stellen"], y=wahl["va"],
                         mode="lines", name="validation",
                         line=dict(width=3, color="crimson")))
fig.add_vline(x=wahl["stelle"], line_dash="dash", line_color="#d8a24a")
fig.update_layout(
    height=430, xaxis_type="log", yaxis_type="log",
    xaxis_title="updates", yaxis_title="error, in scaled units",
    title=f"your choice: {wahl['neuronen']} neurons, learning rate "
          f"{wahl['lernrate']}, frozen after {wahl['stelle']} updates")
fig.show()

In [ ]:
# Its error as a typical miss, next to the clean curve's
miss = np.sqrt(wahl["val"]) * y_s
rule_miss = np.sqrt(np.mean((rein[i_va] - y[i_va]) ** 2))
print(f"validation error {wahl['val']:.3f} scaled,"
      f" typical miss {miss:.1f} kWh/m2a")
print(f"the clean curve on the validation buildings: {rule_miss:.1f} kWh/m2a")
print(f"training range of compactness: {k_lo:.2f} to {k_hi:.2f}")

# Save the chosen network for submission ("Abgabe" = submission)
abgabe = {}
abgabe["neuronen"] = wahl["neuronen"]
abgabe["zustand"] = wahl["zustand"]
abgabe["k_m"] = float(k_m)
abgabe["k_s"] = float(k_s)
abgabe["y_m"] = float(y_m)
abgabe["y_s"] = float(y_s)
torch.save(abgabe, "abgabe_03.pt")
print("written: abgabe_03.pt, submit it together with this notebook")

### Your interpretation

1. **Where you stopped.** Give the point at which your chosen run was frozen, and the reason, read
   off its curve. If the curve is flat, say so and say what that means.
2. **Network size.** Compare the three widths at the same learning rate, for each of your two
   rates, with the numbers. Is the order from best to worst the same at both rates, or do any two
   widths swap places? Widths within 0.002 of each other count as a tie. Say how much of the
   difference you would trust with one split and one seed.
3. **Split, then scale.** The first cell prints the scaling from the training buildings and the
   scaling had it come from all 265. Say what would have leaked, and why your validation error
   would then no longer be an independent check.
4. **The test number.** Write down the typical miss in kWh/m²a you expect on the unseen houses,
   with your reason. Then say what that number will measure once it is computed.
5. **One limit.** Name one kind of building or situation for which that number says nothing, and
   why the evaluation does not cover it.
6. **Design consequence.** A designer wants to use your model to compare two variants of the same
   building. Say what your model could and could not tell them. Refusing, with a reason, is a
   valid answer.

*Your text here.*